In [1]:
import sys, os

sys.path.append('../')
from Libraries import *
from Diagram_Call import *

os.environ["CUDA_VISIBLE_DEVICES"]="1"

import multiprocessing


multiprocessing.set_start_method('spawn', force=True)
       
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
  tf.config.experimental.set_memory_growth(gpu, True)


2025-04-02 08:39:21.818309: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-04-02 08:39:21.818475: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-04-02 08:39:21.819843: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-02 08:39:21.953668: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
def get_flops_block_1(config):
    session = tf.compat.v1.Session()
    graph = tf.compat.v1.get_default_graph()

    with graph.as_default():
        with session.as_default():
            # Build and compile the Block 1 model
            model = DB1T_scipy.get_model_block_1(config)
            model.compile(optimizer='adam', loss='mse')

            # Profile the graph to calculate FLOPS
            run_meta = tf.compat.v1.RunMetadata()
            opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
            flops = tf.compat.v1.profiler.profile(graph=graph, run_meta=run_meta, cmd='scope', options=opts)

    return flops.total_float_ops / 1e12 if flops else 0  # Convert to TFLOPS



In [ ]:
LOOKBACK = 6 # History given as input to the network. Could be modified if needed
GAMMA = 2 # Positive slope of the loss function
NUM_SERV_B1 = 5 # number of services for the block 1
NUM_SERV_H = 1 # number of services for the helper
B = 100 # Number of montecarlo output
SEL_SERVS=[0, 1, 2, 3, 4] #Services to be selected for Helper Block


PHIS = [0.1,0.5,1,10] # Negative slope of the loss function TO BE MODIFIED 
# PHIS = [10]

# cities = ['Dijon', 'Grenoble', 'Lille', 'Lyon', 'Marseille', 'Montpellier', 'Nantes',
#         'Nice', 'Paris','Reims', 'Rennes', 'Strasbourg'] ## Bordeaux and Toulouse are not included in the dataset 

# cities = ['Dijon']
cities = ['Paris']
# ALPHAS=[2,3,5]
ALPHAS=[2]

save=True

DELAY_Block1_Block2 = None # Lenght forecasted by the network. Could be modified if needed
DELAY_Helper = 1 # Lenght forecasted by the network. Could be modified if needed
EPOCHS_block1 = 300 # Number of epochs for the block 1
EPOCHS_block2 = 500 # Number of epochs for the block 2
EPOCHS_Helper = 300 # Number of epochs for the helper
Simulations= 10 # Number of simulations for the optimal window selection


TB_Fpath = 'TRAINING_FLOPS'
save_folder = 'Results_Optimal_Cities_test_kr_ki_new'
# save_folder = 'Results_Optimal_Cities_test_kr_ki_ALLOC_changed_b2'

pair_list=[]
# ppf_static_list=[0.9,0.99]
ppf_static_list=[0.9]
ppf_helper_list=[0.7]

ETAS =[1,2,10,20,30,40,50,70,90,100]
# ETAS = [2]
# ETAS =[20]
